# Food Vision — train and evaluate

Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**.

About 70 minutes: ~10 min to fetch Food-101, ~55 min to train 5 epochs, ~5 min to
evaluate the held-out test split.

At the end this prints three things to copy back into the repo:

1. the SHA-256 of the new checkpoint → `app/config.py`
2. the contents of `evaluation/RESULTS.md`
3. the path to the checkpoint file to upload as a release asset

Unlike the original notebook, the 25,250-image test split is never seen during
training, so the accuracy at the end is real.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader

In [ ]:
# Pull the repo so training uses the same code and the same preprocessing
# transform the API serves.
!git clone --depth 1 https://github.com/vidit-16/foodVision.git /content/foodVision
%cd /content/foodVision
!pip install -q -r requirements.txt

## Dataset

torchvision fetches Food-101 and reads the official `meta/train.json` and
`meta/test.json` split files. This is the step the original notebook skipped.

In [ ]:
from torchvision.datasets import Food101

train = Food101(root="/content/data", split="train", download=True)
test = Food101(root="/content/data", split="test", download=True)

print(f"train images: {len(train):,}")   # 75,750
print(f"test images:  {len(test):,}")    # 25,250
print(f"classes:      {len(train.classes)}")
assert len(train) == 75750 and len(test) == 25250

In [ ]:
# Two minutes, proves the wiring before committing to the full run.
!python training/train.py --data-dir /content/data --smoke-test

## Train

5 epochs, Adam at 1e-4, cross-entropy, AMP, batch size 32 — the recipe from the
original notebook. 10% of the train split is held back for validation; the test
split stays closed.

The best-validation checkpoint is written to `/content/food_vision_resnet50.pt`.

In [ ]:
!python training/train.py \
    --data-dir /content/data \
    --epochs 5 \
    --batch-size 32 \
    --output /content/food_vision_resnet50.pt

## Evaluate

Scores the checkpoint on the 25,250 test images it has never seen, through the
same transform `app/preprocessing.py` applies at inference time.

In [ ]:
import os
os.environ["FOODVISION_WEIGHTS_PATH"] = "/content/food_vision_resnet50.pt"
# The checksum in config.py belongs to the old checkpoint; skip verification
# here and record the new hash below.
os.environ["FOODVISION_WEIGHTS_SHA256"] = ""

!FOODVISION_WEIGHTS_PATH=/content/food_vision_resnet50.pt \
 FOODVISION_WEIGHTS_SHA256= \
 python evaluation/evaluate.py --data-dir /content/data

## Copy these back into the repo

In [ ]:
import hashlib, json, pathlib

path = pathlib.Path("/content/food_vision_resnet50.pt")
digest = hashlib.sha256(path.read_bytes()).hexdigest()

print("=" * 72)
print("1. Paste into app/config.py as WEIGHTS_SHA256:\n")
print(f'   "{digest}"')
print("\n   and bump WEIGHTS_URL to the v1.1.0 release tag.")
print("=" * 72)

results = json.load(open("evaluation/results.json"))
print(f"\n   top-1: {results['top1']*100:.2f}%   top-5: {results['top5']*100:.2f}%")
print(f"   over {results['images']:,} held-out test images")

print("\n" + "=" * 72)
print("2. Replace evaluation/RESULTS.md with everything below:\n")
print("=" * 72)
print(pathlib.Path("evaluation/RESULTS.md").read_text())

In [ ]:
# Save the checkpoint to Drive so it survives the runtime shutting down,
# then download it from there to upload as a GitHub release asset.
from google.colab import drive
drive.mount("/content/drive")

!mkdir -p /content/drive/MyDrive/food_vision_model
!cp /content/food_vision_resnet50.pt /content/drive/MyDrive/food_vision_model/food_vision_resnet50_v1.1.0.pt
!ls -lh /content/drive/MyDrive/food_vision_model/

## Then

1. Create release `v1.1.0` on GitHub, upload `food_vision_resnet50_v1.1.0.pt`
   as the asset, named `food_vision_resnet50.pt`.
2. Update `WEIGHTS_SHA256` and `WEIGHTS_URL` in `app/config.py`.
3. Replace `evaluation/RESULTS.md` with the output above.
4. Update the Status section of the top-level README — the checkpoint is
   validated now, so it can quote the top-1 figure.